# P2C-64 v3.3 (`WACA_G2_V33`: Sparse-Rich Key-14 8ch + Center-10 4ch + 44-Term Pairwise SCReLU + Compact `H=16` MLP) — Google Colab 実行ノートブック

Egaroucid 公開学習データ（約2,551万局面）および既存の v2.0 Phase-1 ベースライン（`raw_ntuple64.p2c`）を再利用し、**P2C-64 v3.3（`WACA_G2_V33`, `H=16` 固定）** を学習・5世代ビット完全一致監査・純スカラー実戦対局（MPC $\sigma$ / 20ms / 50ms / 100ms・`--mo static` vs `--mo b4_hybrid`・`v3.3 vs v3.2` 直接対決）するためのワンクリックノートブックです。

### v3.3 の 4 大革新点
1. **12 次元多項式 Stage 2.5（`WACA_V33_F25 = 12`・追加 popcount 0・追加ロード 0）**：既存 8 次元に 4 個の交差積（`mob_diff*mob_sum`, `mob_diff*v1`, `mob_diff*cmob`, `v1*|v1|`）を追加し、ほぼ 0 ns 追加コストでゲート精度を向上。
2. **0 追加キャッシュミスの Key-14 `8ch` 埋込（`21.02 MiB`）＋ Center-10 `4ch` 埋込（`2.25 MiB`）＋ 44 項 Pairwise・中央×外周カップリング SCReLU ヘッド**：Dense MLP を最良の **`H=16` に固定**したまま、疎パターン側の表現力を抜本強化。
3. **`ply 24..31` 適正化 Flood-Fill ランプ ＋ 0 ロード `outer_rest_sum` 復元**：v3.2 で悪化していた `0-19` / `40-59` の `d4→d8` MPC $\sigma$ を回復。
4. **既存成果物の完全保護**：既存 `cache_bin/`・`run_egaroucid_25m_v20/`・`v30/`・`v31/`・`v32/` は読み取り再利用のみ。v3.3 の全成果物は新規ディレクトリ `run_egaroucid_25m_v33/` へ保存します。

## Cell 1: GPU/VRAM 自動判定 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v33"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
V30_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v30"
V31_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v31"
V32_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v32"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (recommended for Cell 6A / Cell 6B pure-scalar benchmarks & MPC sigma)")

TRAIN_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
MLP_HIDDEN = 16  # Proven optimal low-jitter compact MLP width
print(f"Config  : RUN_NAME={RUN_NAME}, TRAIN_BATCH={TRAIN_BATCH}, MLP_HIDDEN={MLP_HIDDEN}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ 純スカラービルド・単体テスト・32カーネル 0-SIMD ASM監査

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: 既存キャッシュ `train.bin` / `val.bin` および凍結 Phase-1 `raw_ntuple64.p2c` の再利用準備

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

for src, dst in [(cache_tr, local_tr), (cache_va, local_va)]:
    if os.path.exists(dst):
        print("[OK] Local SSD cache present:", dst)
    else:
        assert os.path.exists(src), f"Required binary cache not found on Drive: {src}"
        print(f"[Cache Hit] Copying {src} -> {dst} ...")
        shutil.copy2(src, dst)

# Reuse frozen Phase-1 baseline from run_egaroucid_25m_v20 (0-min Phase 1!)
v20_raw = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
v33_raw = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
local_raw = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"

if os.path.exists(v33_raw):
    print("[OK] Reusing raw_ntuple64.p2c in v3.3 work dir:", v33_raw)
elif os.path.exists(v20_raw):
    print("[Reuse v2.0 Phase-1] Copying frozen raw_ntuple64.p2c -> v3.3 work dir (0-min Phase 1!)...")
    shutil.copy2(v20_raw, v33_raw)
else:
    print("[Fallback] v2.0 raw_ntuple64.p2c not found; running Phase 1 once...")
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_p2c64.py",
        "--train-bin", local_tr,
        "--val-bin", local_va,
        "--work-dir", DRIVE_WORK_DIR,
        "--phases", "phase1",
        "--resume",
        "--batch-size", str(TRAIN_BATCH),
        "--p1-epochs", "6",
    ], check=True)

shutil.copy2(v33_raw, local_raw)
!ls -lh /content/data/

## Cell 4: WACA-G2 v3.3 本学習（`H=16`: 12-Dim Poly Stage 2.5 → 凍結 → Key-14 8ch + Center-10 4ch + 44-Term SCReLU + 32→16→1 MLP + QAT）

In [ ]:
import json

!python3 {REPO_DIR}/python/train_waca_g2_v33.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --raw-p2c {LOCAL_DATA_DIR}/raw_ntuple64.p2c \
    --work-dir {DRIVE_WORK_DIR} \
    --out-name waca_g2_v33.p2c \
    --mlp-hidden {MLP_HIDDEN} \
    --batch-size {TRAIN_BATCH} \
    --s25-epochs 3 \
    --s25-lr 2e-2 \
    --s25-smooth 1e-3 \
    --s3-epochs 7 \
    --s3-qat-epochs 2 \
    --qat-lr-mult 0.12 \
    --qat-lr-min 0.02 \
    --s3-emb-lr 8e-3 \
    --s3-mlp-lr 4e-3 \
    --s3-ply-smooth 1.2e-3 \
    --s3-blk-smooth 5.0e-3 \
    --resume

print("\nSaved v3.3 artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics_v33.*
print("\n=== metrics_v33.json ===")
with open(f"{DRIVE_WORK_DIR}/metrics_v33.json") as f:
    print(json.dumps(json.load(f), indent=2))

## Cell 5: C++/Python 全3段ビット完全一致（`S1=0 S2.5=0 S3=0`）＆ 5世代後方互換・16対称（$D_4 \times \mathbb{Z}_2$）監査

In [ ]:
import os
import subprocess

models = [f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c", f"{DRIVE_WORK_DIR}/waca_g2_v33.p2c"]
for cand in [
    f"{V32_RUN_DIR}/waca_g2_v32.p2c",
    f"{V31_RUN_DIR}/waca_g2_v31.p2c",
    f"{V30_RUN_DIR}/waca_g2.p2c",
]:
    if os.path.exists(cand):
        models.append(cand)

val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"
subprocess.run([
    "python3", f"{REPO_DIR}/python/test_bit_exact_io.py",
    "--data", val_bin,
    "--num-pos", "1000",
    "--models", *models,
    "--bench-bridge",
], check=True)

## Cell 6A: （CPUランタイム推奨・前半）MPC $\sigma$ ＋ 20ms 対局（`--mo static` / `--mo b4_hybrid` / `v3.3 vs v3.2` 直接対決）

GPUを切断し CPU ランタイムへ切り替えた後、**Cell 1 → Cell 2 → Cell 6A** を実行してください。冒頭の `TT_GUARD` で `tt_guard` 値を自在に指定できます（`None` なら `metrics_v33.json` の `recommended_tt_guard` を自動採用、未生成なら `5.5`）。各対局の完了直後に CSV 最終行を `print` します：
1. **MPC $\sigma$（`d2→d6`, `d4→d8`, 2,000局面）**：`RAW_NTUPLE_64` vs `WACA_G2_V32`（存在時） vs `WACA_G2_V33`
2. **`v3.3 vs RAW`（20ms, 1,000局, `--mo static`, `wide_safe_bnd 2.5/1.5`）**
3. **`v3.3 vs RAW`（20ms, 1,000局, `--mo b4_hybrid`, `wide_safe_bnd 2.5/1.5`）**
4. **`v3.3 vs v3.2` 直接対決（20ms, 1,000局, `--mo b4_hybrid`, `wide_safe_bnd 2.5/1.5`, v3.2存在時自動実行）**

In [ ]:
import json
import os
import subprocess

# Configurable TT guard (set e.g. "5.5" or "6.0" to override; None = auto from metrics_v33.json recommended_tt_guard)
TT_GUARD = None

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V33_P2C = f"{DRIVE_WORK_DIR}/waca_g2_v33.p2c"
V32_P2C = f"{V32_RUN_DIR}/waca_g2_v32.p2c"
CSV_V33 = f"{DRIVE_WORK_DIR}/match_results_v33.csv"
METRICS_JSON = f"{DRIVE_WORK_DIR}/metrics_v33.json"
val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

if TT_GUARD is None:
    if os.path.exists(METRICS_JSON):
        with open(METRICS_JSON) as f:
            m_js = json.load(f)
        TT_GUARD = f"{m_js.get('residual_stats', {}).get('recommended_tt_guard', 5.5):.2f}"
    else:
        TT_GUARD = "5.5"
else:
    TT_GUARD = str(TT_GUARD)
print(f"[Config 6A] Using TT_GUARD = {TT_GUARD}", flush=True)

assert os.path.exists(RAW_P2C), f"Missing RAW model: {RAW_P2C}"
assert os.path.exists(V33_P2C), f"Missing v3.3 model: {V33_P2C}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

# 1) MPC sigma measurement (d2->d6 and d4->d8, 2,000 pos): RAW vs v3.2 vs v3.3
mpc_models = [RAW_P2C]
if os.path.exists(V32_P2C):
    mpc_models.append(V32_P2C)
mpc_models.append(V33_P2C)

for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== P2C-64 v3.3 MPC Sigma: depth {sh} -> {dp} (2,000 pos) ===", flush=True)
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--gate-mode", "wide_safe_bnd",
        "--m25", "2.5",
        "--m3", "1.5",
        "--tt-guard", TT_GUARD,
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_v33_d{sh}_d{dp}.csv",
        "--models", *mpc_models,
    ], check=True)

# 2) 20ms Fair Matches: WACA_G2_V33 vs RAW_NTUPLE_64 (static & b4_hybrid, 1,000 games each)
suite_6a = [
    (f"[6A-1] 20ms STATIC    (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 1000g)", "static"),
    (f"[6A-2] 20ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 1000g)", "b4_hybrid"),
]
for label, mo in suite_6a:
    run_match_and_log(
        f"{label}: WACA_G2_V33 vs RAW_NTUPLE_64",
        [
            MATCH_BIN, "--p1", V33_P2C, "--p2", RAW_P2C,
            "--mode", "time", "--limit", "20", "--pairs", "500",
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", mo, "--gate-mode", "wide_safe_bnd",
            "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
            "--seed", "20261127", "--csv", CSV_V33,
        ],
        CSV_V33,
    )

# 3) Head-to-head v3.3 vs v3.2 (20ms, 1,000 games, b4_hybrid)
if os.path.exists(V32_P2C):
    run_match_and_log(
        f"[6A-3] Head-to-Head 20ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 1000g): WACA_G2_V33 vs WACA_G2_V32",
        [
            MATCH_BIN, "--p1", V33_P2C, "--p2", V32_P2C,
            "--mode", "time", "--limit", "20", "--pairs", "500",
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", "b4_hybrid", "--gate-mode", "wide_safe_bnd",
            "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
            "--seed", "20261127", "--csv", CSV_V33,
        ],
        CSV_V33,
    )

print(f"\n[Cell 6A Done] Results saved to: {CSV_V33}")

## Cell 6B: （CPUランタイム推奨・後半）超深読み 50ms / 100ms 検証（各 600 局・`--mo b4_hybrid` / `--mo static`）

セッション切断に強い後半専用セルです。冒頭の `TT_GUARD` 変数を変更することで任意のガード幅を再検証できます（`None` なら `metrics_v33.json` の推奨値を自動採用）。50ms（各 600 局）および 100ms（各 600 局）の超深読み帯で `b4_hybrid` / `static` を計測し、毎試合ごとに CSV 最終行を表示します。

In [ ]:
import json
import os
import subprocess

# Configurable TT guard (set e.g. "5.5" or "6.0" to override; None = auto from metrics_v33.json recommended_tt_guard)
TT_GUARD = None

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V33_P2C = f"{DRIVE_WORK_DIR}/waca_g2_v33.p2c"
CSV_V33 = f"{DRIVE_WORK_DIR}/match_results_v33.csv"
METRICS_JSON = f"{DRIVE_WORK_DIR}/metrics_v33.json"

if TT_GUARD is None:
    if os.path.exists(METRICS_JSON):
        with open(METRICS_JSON) as f:
            m_js = json.load(f)
        TT_GUARD = f"{m_js.get('residual_stats', {}).get('recommended_tt_guard', 5.5):.2f}"
    else:
        TT_GUARD = "5.5"
else:
    TT_GUARD = str(TT_GUARD)
print(f"[Config 6B] Using TT_GUARD = {TT_GUARD}", flush=True)

assert os.path.exists(RAW_P2C), f"Missing RAW model: {RAW_P2C}"
assert os.path.exists(V33_P2C), f"Missing v3.3 model: {V33_P2C}"

def run_match_and_log(label, cmd, csv_path):
    print(f"\n=== {label} ===", flush=True)
    subprocess.run(cmd, check=True)
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if len(lines) >= 2:
            print(f"[CSV LAST ROW] {lines[-1]}", flush=True)

suite_6b = [
    (f"[6B-1] 50ms  B4_HYBRID (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 600g)", "50",  "300", "b4_hybrid"),
    (f"[6B-2] 50ms  STATIC    (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 600g)", "50",  "300", "static"),
    (f"[6B-3] 100ms B4_HYBRID (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 600g)", "100", "300", "b4_hybrid"),
    (f"[6B-4] 100ms STATIC    (wide_safe_bnd 2.5/1.5 g={TT_GUARD}, 600g)", "100", "300", "static"),
]

for label, limit, pairs, mo in suite_6b:
    run_match_and_log(
        f"{label}: WACA_G2_V33 vs RAW_NTUPLE_64",
        [
            MATCH_BIN, "--p1", V33_P2C, "--p2", RAW_P2C,
            "--mode", "time", "--limit", limit, "--pairs", pairs,
            "--opening-plies", "8", "--endgame-empties", "10", "--tt-mb", "256",
            "--mpc", "--mpc-t", "1.5", "--no-mpc-pure-v64",
            "--mo", mo, "--gate-mode", "wide_safe_bnd",
            "--m25", "2.5", "--m3", "1.5", "--tt-guard", TT_GUARD,
            "--seed", "20261127", "--csv", CSV_V33,
        ],
        CSV_V33,
    )

print(f"\n=== [Cell 6B Done] Full CSV Summary ({CSV_V33}) ===")
!cat {CSV_V33}